# Argumenten

# Argument 1
New York heeft meer criminaliteit dan Los Angeles
## Perspectief
New York heeft een zeer hoge bevolkingsdichtheid, waardoor veel mensen dagelijks dicht op elkaar leven en werken. Dit vergroot de kans op criminaliteit, zoals diefstal, zakkenrollerij en andere veelvoorkomende misdrijven. Daarnaast trekt de stad jaarlijks miljoenen toeristen aan, wat extra mogelijkheden biedt voor bepaalde vormen van criminaliteit. Ook spelen sociaaleconomische problemen, zoals armoede en bendecriminaliteit in sommige wijken, een rol. Door de grote omvang, drukte en constante stroom van inwoners en bezoekers is het voor politie en andere instanties een uitdaging om criminaliteit volledig te voorkomen.
# Argument 2
De verschillen in criminaliteit tussen wijken zijn groot, zowel in New York als in Los Angeles.
## Perspectief
Criminaliteit is niet gelijk verdeeld over een stad. Sommige wijken kennen relatief weinig misdrijven, terwijl andere buurten juist vaker met criminaliteit te maken hebben. Factoren zoals inkomen, werkgelegenheid, onderwijsniveau en de beschikbaarheid van voorzieningen kunnen hierbij een belangrijke rol spelen. Zowel in New York als in Los Angeles bestaan grote sociaaleconomische verschillen tussen wijken. Gebieden waarin meer wordt geïnvesteerd in openbare voorzieningen, sociale programma’s en veiligheid laten vaak lagere criminaliteitscijfers zien. Daardoor geeft een gemiddeld criminaliteitscijfer voor een hele stad niet altijd een nauwkeurig beeld van de veiligheid op wijkniveau.

# Visualisaties 

In [11]:
import plotly.graph_objs as go
import plotly.express as px
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

df_nypd = pd.read_csv("../data/processed/crime_ny_processed.csv")
df_lapd = pd.read_csv("../data/processed/crime_la_processed.csv")


In [12]:
nypd_per_jaar = df_nypd.groupby('year').size().reset_index(name='count')
nypd_per_jaar['city'] = 'NYC (NYPD)'

lapd_per_jaar = df_lapd.groupby('year').size().reset_index(name='count')
lapd_per_jaar['city'] = 'LA (LAPD)'

df_combined = pd.concat([nypd_per_jaar, lapd_per_jaar])

jaren = [2020, 2021, 2022, 2023]

nypd_vals = nypd_per_jaar.set_index('year').reindex(jaren)['count']
lapd_vals = lapd_per_jaar.set_index('year').reindex(jaren)['count']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=jaren,
    y=nypd_vals,
    name='NYC (NYPD)',
    marker_color='#1f77b4',
    text=[f'{int(v):,}' for v in nypd_vals],
    textposition='outside'
))

fig.add_trace(go.Bar(
    x=jaren,
    y=lapd_vals,
    name='LA (LAPD)',
    marker_color='#ff7f0e',
    text=[f'{int(v):,}' for v in lapd_vals],
    textposition='outside'
))

fig.update_layout(
    barmode='group',
    title=dict(text='Misdaden per jaar: NYC vs LA (2020–2024)', font=dict(size=20, family='Arial Black')),
    xaxis_title='Jaar',
    yaxis_title='Aantal misdaden / arrestaties',
    xaxis=dict(tickmode='array', tickvals=jaren),
    yaxis=dict(tickformat=','),
    legend=dict(title=''),
    template='plotly_white',
    width=900,
    height=600
)

fig.show()

In [13]:
df_nypd['year_month'] = pd.to_datetime(df_nypd['arrest_date']).dt.to_period('M')
df_lapd['year_month'] = pd.to_datetime(df_lapd['date_occ']).dt.to_period('M')

nypd_maand = df_nypd.groupby('year_month').size().reset_index(name='count')
lapd_maand = df_lapd.groupby('year_month').size().reset_index(name='count')

nypd_maand['year_month'] = nypd_maand['year_month'].dt.to_timestamp()
lapd_maand['year_month'] = lapd_maand['year_month'].dt.to_timestamp()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=nypd_maand['year_month'],
    y=nypd_maand['count'],
    mode='lines',
    name='NYC (NYPD)',
    line=dict(color='#1f77b4', width=2)
))

fig.add_trace(go.Scatter(
    x=lapd_maand['year_month'],
    y=lapd_maand['count'],
    mode='lines',
    name='LA (LAPD)',
    line=dict(color='#ff7f0e', width=2)
))

fig.update_layout(
    title=dict(text='Misdaden per maand: NYC vs LA (2020–2023)', font=dict(size=20, family='Arial Black')),
    xaxis_title='Periode',
    yaxis_title='Aantal misdaden / arrestaties',
    xaxis=dict(
        tickmode='array',
        tickvals=pd.date_range('2020-01-01', '2024-01-01', freq='YS'),
        ticktext=[str(y) for y in range(2020, 2025)]
    ),
    yaxis=dict(tickformat=','),
    legend=dict(title=''),
    template='plotly_white',
    width=1100,
    height=500
)

fig.show()

In [14]:
nypd_types = (df_nypd['ofns_desc']
              .value_counts()
              .head(10)
              .reset_index())
nypd_types.columns = ['misdaad', 'count']

lapd_types = (df_lapd['crm_cd_desc']
              .value_counts()
              .head(10)
              .reset_index())
lapd_types.columns = ['misdaad', 'count']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top 10 misdaden — NYC (NYPD)', 'Top 10 misdaden — LA (LAPD)')
)

fig.add_trace(go.Bar(
    x=nypd_types['count'][::-1],
    y=nypd_types['misdaad'][::-1],
    orientation='h',
    marker_color='#1f77b4',
    text=[f'{int(v):,}' for v in nypd_types['count'][::-1]],
    textposition='outside',
    name='NYC (NYPD)'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=lapd_types['count'][::-1],
    y=lapd_types['misdaad'][::-1],
    orientation='h',
    marker_color='#ff7f0e',
    text=[f'{int(v):,}' for v in lapd_types['count'][::-1]],
    textposition='outside',
    name='LA (LAPD)'
), row=1, col=2)

fig.update_xaxes(title_text='Aantal arrestaties', tickformat=',', row=1, col=1)
fig.update_xaxes(title_text='Aantal meldingen', tickformat=',', row=1, col=2)

fig.update_layout(
    showlegend=False,
    template='plotly_white',
    width=1400,
    height=700,
    title=dict(font=dict(size=20, family='Arial Black'))
)

fig.show()

In [15]:
nyc_borough_coords = {
    'MANHATTAN': {'lat': 40.7831, 'lon': -73.9712},
    'BRONX': {'lat': 40.8448, 'lon': -73.8648},
    'BROOKLYN': {'lat': 40.6782, 'lon': -73.9442},
    'QUEENS': {'lat': 40.7282, 'lon': -73.7949},
    'STATEN ISLAND': {'lat': 40.5795, 'lon': -74.1502}
}

# NYPD gebruikt vaak afkortingen: K, Q, M, B, S
nyc_borough_map = {
    'K': 'BROOKLYN', 'Q': 'QUEENS', 'M': 'MANHATTAN',
    'B': 'BRONX', 'S': 'STATEN ISLAND'
}
df_nypd['borough_naam'] = df_nypd['arrest_boro'].map(nyc_borough_map).fillna(df_nypd['arrest_boro'])

nypd_coords = pd.DataFrame([
    {'borough_naam': naam, 'Latitude': info['lat'], 'Longitude': info['lon']}
    for naam, info in nyc_borough_coords.items()
])

In [16]:
la_area_coords = {
    'Central': {'lat': 34.0445, 'lon': -118.2460},
    'Rampart': {'lat': 34.0586, 'lon': -118.2823},
    'Southwest': {'lat': 34.0141, 'lon': -118.3220},
    'Hollenbeck': {'lat': 34.0392, 'lon': -118.2080},
    'Harbor': {'lat': 33.7570, 'lon': -118.2920},
    'Hollywood': {'lat': 34.0980, 'lon': -118.3267},
    'Wilshire': {'lat': 34.0593, 'lon': -118.3470},
    'West LA': {'lat': 34.0461, 'lon': -118.4500},
    'Van Nuys': {'lat': 34.1870, 'lon': -118.4480},
    'West Valley': {'lat': 34.1900, 'lon': -118.5290},
    'Northeast': {'lat': 34.1062, 'lon': -118.2280},
    '77th Street': {'lat': 33.9700, 'lon': -118.2890},
    'Newton': {'lat': 34.0090, 'lon': -118.2540},
    'Pacific': {'lat': 33.9930, 'lon': -118.4280},
    'N Hollywood': {'lat': 34.1870, 'lon': -118.3790},
    'Foothill': {'lat': 34.2570, 'lon': -118.3700},
    'Devonshire': {'lat': 34.2520, 'lon': -118.5350},
    'Southeast': {'lat': 33.9380, 'lon': -118.2480},
    'Mission': {'lat': 34.2720, 'lon': -118.4490},
    'Olympic': {'lat': 34.0610, 'lon': -118.3050},
    'Topanga': {'lat': 34.1980, 'lon': -118.6050}
}

lapd_coords = pd.DataFrame([
    {'area_name': naam, 'LAT': info['lat'], 'LON': info['lon']}
    for naam, info in la_area_coords.items()
])

In [ ]:
# --- Top 5 + overig per stad bepalen ---
def top5_plus_overig(df, col):
    top5 = df[col].value_counts().head(5).index.tolist()
    return df[col].apply(lambda x: x if x in top5 else 'Overig')

df_nypd['categorie'] = top5_plus_overig(df_nypd, 'ofns_desc')
df_lapd['categorie'] = top5_plus_overig(df_lapd, 'crm_cd_desc')

# Vaste kleuren per categorie zodat alle pies dezelfde legenda gebruiken
alle_categorieen = sorted(set(df_nypd['categorie'].unique()) | {'Overig'})
kleuren_nypd = px.colors.qualitative.Set3
categorie_kleuren_nypd = {cat: kleuren_nypd[i % len(kleuren_nypd)]
                          for i, cat in enumerate(sorted(df_nypd['categorie'].unique()))}
categorie_kleuren_lapd = {cat: kleuren_nypd[i % len(kleuren_nypd)]
                           for i, cat in enumerate(sorted(df_lapd['categorie'].unique()))}

In [18]:
print(df_nypd['arrest_boro'].unique())
print(df_lapd['area_name'].unique())

<StringArray>
['K', 'B', 'M', 'S', 'Q']
Length: 5, dtype: str
<StringArray>
['N Hollywood',    'Van Nuys',    'Wilshire',     'Pacific',  'Hollenbeck',
   'Southwest',   'Northeast',  'Devonshire',     'Topanga',   'Hollywood',
     'Olympic',   'Southeast',      'Newton',    'Foothill',     'Mission',
     'Rampart',     'Central',     'West LA', '77th Street', 'West Valley',
      'Harbor']
Length: 21, dtype: str


In [19]:
# NYC mapping toepassen
nyc_borough_map = {
    'K': 'BROOKLYN', 'Q': 'QUEENS', 'M': 'MANHATTAN',
    'B': 'BRONX', 'S': 'STATEN ISLAND'
}
df_nypd['borough_naam'] = df_nypd['arrest_boro'].map(nyc_borough_map)

# Check of alles gemapt is (geen NaN)
print(df_nypd['borough_naam'].isna().sum())  # moet 0 zijn

0


In [20]:
print(set(df_lapd['area_name'].unique()) - set(la_area_coords.keys()))  # moet leeg zijn: set()

set()


In [21]:
# Pie-data per wijk per categorie
nypd_pie_data = df_nypd.groupby(['borough_naam', 'categorie']).size().reset_index(name='count')
lapd_pie_data = df_lapd.groupby(['area_name', 'categorie']).size().reset_index(name='count')

In [22]:
# --- Top 5 + overig per stad bepalen ---
def top5_plus_overig(df, col):
    top5 = df[col].value_counts().head(5).index.tolist()
    return df[col].apply(lambda x: x if x in top5 else 'Overig')

df_nypd['categorie'] = top5_plus_overig(df_nypd, 'ofns_desc')
df_lapd['categorie'] = top5_plus_overig(df_lapd, 'crm_cd_desc')


# Vaste kleuren per categorie zodat alle pies dezelfde legenda gebruiken
alle_categorieen = sorted(set(df_nypd['categorie'].unique()) | {'Overig'})
kleuren_nypd = px.colors.qualitative.Set3
categorie_kleuren_nypd = {cat: kleuren_nypd[i % len(kleuren_nypd)]
                          for i, cat in enumerate(sorted(df_nypd['categorie'].unique()))}
categorie_kleuren_lapd = {cat: kleuren_nypd[i % len(kleuren_nypd)]
                           for i, cat in enumerate(sorted(df_lapd['categorie'].unique()))}

In [23]:
def maak_donut_grid_kaart(pie_data, coords, wijk_col, kleuren_dict, titel, cols=3):
    wijken = coords[wijk_col].tolist()
    n_wijken = len(wijken)
    rows = int(np.ceil(n_wijken / cols))

    specs = [[{'type': 'domain'} for _ in range(cols)] for _ in range(rows)]
    fig = make_subplots(
        rows=rows, cols=cols, specs=specs,
        subplot_titles=wijken,
        horizontal_spacing=0.05,
        vertical_spacing=0.15
    )

    for i, wijk in enumerate(wijken):
        row = i // cols + 1
        col = i % cols + 1
        wijk_data = pie_data[pie_data[wijk_col] == wijk]

        fig.add_trace(go.Pie(
            labels=wijk_data['categorie'],
            values=wijk_data['count'],
            hole=0.4,
            name=wijk,
            marker=dict(colors=[kleuren_dict.get(c, '#cccccc') for c in wijk_data['categorie']]),
            showlegend=(i == 0),
            textposition='inside',
            textinfo='percent'
        ), row=row, col=col)

    fig.update_layout(
        title=titel,
        height=400*rows,
        width=350*cols,
        showlegend=True
    )
    return fig

fig_nyc_c = maak_donut_grid_kaart(
    nypd_pie_data, nypd_coords, 'borough_naam',
    categorie_kleuren_nypd, 'NYC — Misdaadcategorieën per Borough', cols=3
)
fig_nyc_c.show()

fig_la_c = maak_donut_grid_kaart(
    lapd_pie_data, lapd_coords, 'area_name',
    categorie_kleuren_lapd, 'LA — Misdaadcategorieën per Area', cols=5
)
fig_la_c.show()